# Targeted chunk review and leakage gate

The 512-token reference plan is the working candidate. This notebook identifies
the smaller review scope, publishes independent exact chunk holds and trials a
targeted review protocol. It preserves all original text and inherited partitions.

## Bind the reference plan

Use the verified Notebook 05 handoff and the Notebook 06 reference snapshot. The
development pilot demonstrated how the window review workflow ran and where to
concentrate review. The pilot selection cannot estimate a corpus error rate.

In [ ]:
import json
from pathlib import Path

import nbformat
import pandas as pd
from IPython.display import display
from pyspark import StorageLevel
from pyspark.sql import Row, Window
from pyspark.sql import functions as F

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "notebooks/06_chunk_evaluation.ipynb").is_file()
)
source = nbformat.read(ROOT / "notebooks/06_chunk_evaluation.ipynb", as_version=4)
cells = {cell.id: cell.source for cell in source.cells if cell.cell_type == "code"}
upstream = {"__name__": "__main__"}
for cell_id in (
    "06-configure",
    "06-bind",
    "06-review-queue",
    "06-review-protocol",
    "06-token-policy",
    "06-token-materialize",
    "06-token-review-gate",
):
    exec(compile(cells[cell_id], f"06_chunk_evaluation.ipynb:{cell_id}", "exec"), upstream)

spark = upstream["spark"]
parents = upstream["parents"]
plans = upstream["token_plans"]
reference_audit = upstream["token_audit"]
reference_dir = upstream["token_dir"]
handoff = upstream["HANDOFF"]
sha256_file = upstream["sha256_file"]
canonical_sha256 = upstream["canonical_sha256"]
make_review_packets = upstream["make_review_packets"]
replay_reference = upstream["validate_reference_decisions"]
encoding = upstream["REFERENCE_ENCODING"]
budget = upstream["REFERENCE_CAP"]
review_policy = {
    "version": "CHUNK_POLICY_V1_CANDIDATE",
    "reference_manifest_sha256": sha256_file(reference_dir / "MANIFEST.json"),
    "reference_encoding": encoding,
    "reference_content_tokens": budget,
    "nonparagraph_sample_fraction": 0.25,
    "paragraph_sample_parents": 500,
    "selection": "split_format_length_risk_stratified_stable_hash",
    "semantic_outcomes": ["none", "boundary_preference", "material_evidence_break", "uncertain"],
    "release_rule": "Targeted review and paragraph sample remain pending until evaluated.",
    "escalation_rule": "Expand any affected stratum if a material or uncertain case is found.",
    "fallback_rule": (
        "Use audited paragraph blocks; hold over-budget blocks from token-limited use."
    ),
    "exact_leakage_rule": "Hold all occurrences of every exact hash appearing across splits.",
}
CODEPOINT_WINDOW = 4_800
CONTEXT_WIDTH = 320
output = handoff.with_name(f"{handoff.name}_chunk_policy_candidate_v1")


## Measure the risk scope and freeze stratified samples

Classify each parent using its internal reference boundary kinds. Sentence,
whitespace, grapheme and code-point cuts enter the nonparagraph scope. Format
is a sampling stratum. The semantic samples cover 25% of the nonparagraph parents
and 500 paragraph or terminal-only parents. Every stratum has representation.
A decision records whether the proposed boundary breaks evidence or merely affects readability.

In [ ]:
import math

boundary_roster = (
    plans.withColumn("chunk_count", F.size("plan.ends"))
    .withColumn(
        "nonparagraph_offsets",
        F.transform(
            F.filter(
                F.arrays_zip(
                    F.col("plan.ends").alias("end"), F.col("plan.boundary_kinds").alias("kind")
                ),
                lambda item: (item.end < F.col("clean_chars")) & (item.kind != "paragraph"),
            ),
            lambda item: item.end,
        ),
    )
    .withColumn(
        "risk",
        F.when(
            F.exists("plan.boundary_kinds", lambda kind: kind.isin("grapheme", "codepoint")),
            "grapheme_or_codepoint",
        )
        .when(F.array_contains("plan.boundary_kinds", "whitespace"), "whitespace")
        .when(F.size("nonparagraph_offsets") > 0, "sentence")
        .when(F.col("chunk_count") > 1, "paragraph")
        .otherwise("terminal_only"),
    )
    .withColumn(
        "length_band",
        F.when(F.col("clean_chars") <= 1200, "up_to_1200")
        .when(F.col("clean_chars") <= 4800, "1201_to_4800")
        .when(F.col("clean_chars") <= 9600, "4801_to_9600")
        .otherwise("over_9600"),
    )
    .withColumn(
        "target_windows",
        F.array_distinct(
            F.transform(
                "nonparagraph_offsets", lambda end: F.floor(end / CODEPOINT_WINDOW).cast("int")
            )
        ),
    )
    .persist(StorageLevel.DISK_ONLY)
)

risk_population = boundary_roster.where(F.size("nonparagraph_offsets") > 0)
structural_population = boundary_roster.where(F.size("nonparagraph_offsets") == 0)
risk_stats = (
    risk_population.agg(
        F.count("*").alias("parents"),
        F.sum(F.size("nonparagraph_offsets")).alias("internal_nonparagraph_boundaries"),
        F.sum(F.size("target_windows")).alias("target_windows"),
        F.sum(F.ceil(F.col("clean_chars") / CODEPOINT_WINDOW)).alias("whole_parent_windows"),
    )
    .first()
    .asDict()
)
structural_stats = {
    row.risk: row["count"] for row in structural_population.groupBy("risk").count().collect()
}
if risk_stats["parents"] + sum(structural_stats.values()) != reference_audit["parents"]:
    raise ValueError("The boundary risk strata did not reconcile to the reference corpus.")

strata_columns = ["ml_split", "format", "length_band", "risk"]


def stratified_draw(population, target, label):
    counts = population.groupBy(*strata_columns).count().orderBy(*strata_columns).collect()
    if not len(counts) <= target <= sum(row["count"] for row in counts):
        raise ValueError("The sample target must cover every nonempty stratum.")
    mandatory = [row["count"] if row.risk == "grapheme_or_codepoint" else 1 for row in counts]
    remainder = target - sum(mandatory)
    capacity = [row["count"] - m for row, m in zip(counts, mandatory, strict=True)]
    ideals = (
        [remainder * room / sum(capacity) for room in capacity] if remainder else [0] * len(counts)
    )
    quotas = [base + math.floor(ideal) for base, ideal in zip(mandatory, ideals, strict=True)]
    order = sorted(range(len(counts)), key=lambda i: (-(ideals[i] % 1), i))
    for i in order[: target - sum(quotas)]:
        quotas[i] += 1
    overallocated = any(quota > row["count"] for quota, row in zip(quotas, counts, strict=True))
    if sum(quotas) != target or overallocated:
        raise ValueError("The stratum quotas do not cover the requested sample.")
    allocations = spark.createDataFrame(
        [
            (*[row[key] for key in strata_columns], row["count"], quota)
            for row, quota in zip(counts, quotas, strict=True)
        ],
        [*strata_columns, "stratum_parents", "stratum_sample"],
    )
    ordering = Window.partitionBy(*strata_columns).orderBy("draw_key", "release_row")
    return (
        population.join(F.broadcast(allocations), strata_columns)
        .withColumn(
            "draw_key",
            F.sha2(
                F.concat_ws("|", F.lit(review_policy["version"]), F.lit(label), "native_identity"),
                256,
            ),
        )
        .withColumn("stratum_rank", F.row_number().over(ordering))
        .where(F.col("stratum_rank") <= F.col("stratum_sample"))
        .withColumn("sample", F.lit(label))
    )


select_columns = [
    "release_row",
    "native_identity",
    "clean_sha256",
    "ml_split",
    "format",
    "length_band",
    "risk",
    "clean_chars",
    "chunk_count",
    "nonparagraph_offsets",
    "target_windows",
    "stratum_parents",
    "stratum_sample",
    "sample",
    "draw_key",
]
risk_sample = stratified_draw(
    risk_population,
    round(risk_stats["parents"] * review_policy["nonparagraph_sample_fraction"]),
    "nonparagraph",
).select(*select_columns)
paragraph_sample = stratified_draw(
    structural_population, review_policy["paragraph_sample_parents"], "paragraph_control"
).select(*select_columns)
sampled = risk_sample.unionByName(paragraph_sample).persist(StorageLevel.DISK_ONLY)
sample_stats = [
    row.asDict()
    for row in sampled.groupBy("sample")
    .agg(
        F.count("*").alias("parents"),
        F.sum(F.size("target_windows")).alias("nonparagraph_windows"),
        F.sum(F.ceil(F.col("clean_chars") / CODEPOINT_WINDOW)).alias("whole_parent_windows"),
    )
    .orderBy("sample")
    .collect()
]
print("Nonparagraph boundary scope", risk_stats)
print("Paragraph and terminal scopes", structural_stats)
display(pd.DataFrame(sample_stats))


## Apply the exact chunk leakage gate

Hash the exact reference slices in Spark. Hold all occurrences of an exact
cross-split group from model-facing use while retaining the source text, offsets
and assigned splits. Reconcile the hold ledger with the previous reference audit.
The long subgroup is included in the uncovered count. The gate runs again on any
changed boundary plan before downstream use.

In [ ]:
if output.exists():
    saved_manifest = json.loads((output / "MANIFEST.json").read_text(encoding="utf-8"))
    saved_policy = output / "CHUNK_POLICY_V1_CANDIDATE.json"
    if sha256_file(saved_policy) != saved_manifest["policy_sha256"]:
        raise ValueError("The saved leakage policy receipt failed verification.")
    previous = json.loads(saved_policy.read_text(encoding="utf-8"))
    if previous["reference_manifest_sha256"] != review_policy["reference_manifest_sha256"]:
        raise ValueError("The saved leakage audit belongs to a different reference plan.")
    exact_groups = upstream["bound_review_table"](
        output, "exact_chunk_groups", saved_manifest["tables"]["exact_chunk_groups"]
    )
    holds = upstream["bound_review_table"](
        output, "exact_chunk_holds", saved_manifest["tables"]["exact_chunk_holds"]
    )
    leakage_stats = previous["exact_leakage"]
    remaining_cross_split = previous["remaining_exact_cross_split_groups_in_reference_after_holds"]
else:
    split_bit = (
        F.when(F.col("ml_split") == "train", 1)
        .when(F.col("ml_split") == "validation", 2)
        .when(F.col("ml_split") == "test", 4)
    )
    slices = (
        plans.select(
            "release_row",
            "native_identity",
            "release_group_id",
            "ml_split",
            "clean_sha256",
            "cross_split_review_required",
            "plan.ends",
            F.posexplode("plan.ends").alias("chunk_ordinal", "end"),
        )
        .withColumn("start", F.coalesce(F.get("ends", F.col("chunk_ordinal") - 1), F.lit(0)))
        .withColumn("codepoints", F.col("end") - F.col("start"))
        .drop("ends")
    )
    occurrences = (
        slices.join(
            parents.select("release_row", "clean_sha256", "clean_text"),
            ["release_row", "clean_sha256"],
        )
        .withColumn(
            "chunk_sha256", F.sha2(F.substring("clean_text", F.col("start") + 1, "codepoints"), 256)
        )
        .drop("clean_text")
        .withColumn("split_bit", split_bit)
        .persist(StorageLevel.DISK_ONLY)
    )
    verified_chunk_count = occurrences.count()
    if verified_chunk_count != reference_audit["chunks"]:
        raise ValueError("The chunk occurrence roster differs from the reference snapshot.")
    no_parent_overlap = ~F.col("cross_split_review_required")
    exact_groups = (
        occurrences.groupBy("chunk_sha256", "codepoints")
        .agg(
            F.bit_or("split_bit").alias("split_mask"),
            F.bit_or(F.when(no_parent_overlap, F.col("split_bit")).otherwise(0)).alias(
                "comparison_split_mask"
            ),
            F.count("*").alias("occurrences"),
            F.count_if(no_parent_overlap).alias("comparison_occurrences"),
        )
        .where(F.col("split_mask").isin(3, 5, 6, 7))
        .withColumn("uncovered_by_parent_overlap", F.col("comparison_split_mask").isin(3, 5, 6, 7))
        .withColumn(
            "long_uncovered", F.col("uncovered_by_parent_overlap") & (F.col("codepoints") >= 500)
        )
        .persist(StorageLevel.DISK_ONLY)
    )
    holds = occurrences.join(
        F.broadcast(exact_groups.select("chunk_sha256", "codepoints")),
        ["chunk_sha256", "codepoints"],
    ).withColumn("hold_reason", F.lit("exact_chunk_in_multiple_splits"))
    leakage_stats = (
        exact_groups.agg(
            F.count("*").alias("cross_split_hashes"),
            F.sum("occurrences").alias("cross_split_occurrences"),
            F.count_if("uncovered_by_parent_overlap").alias(
                "cross_split_hashes_without_known_parent_overlap"
            ),
            F.count_if(F.col("codepoints") >= 500).alias(
                "cross_split_hashes_at_least_500_codepoints"
            ),
            F.count_if("long_uncovered").alias(
                "long_cross_split_hashes_without_known_parent_overlap"
            ),
            F.sum(
                F.when(
                    F.col("uncovered_by_parent_overlap"), F.col("comparison_occurrences")
                ).otherwise(0)
            ).alias("no_parent_overlap_occurrences"),
            F.sum(
                F.when(F.col("long_uncovered"), F.col("comparison_occurrences")).otherwise(0)
            ).alias("long_no_parent_overlap_occurrences"),
        )
        .first()
        .asDict()
    )
    if any(
        leakage_stats[key] != value for key, value in reference_audit["exact_chunk_reuse"].items()
    ):
        raise ValueError("Exact chunk reuse counts differ from the reference audit.")
    hold_keys = holds.select("release_row", "chunk_ordinal")
    held_counts = hold_keys.agg(
        F.count("*"), F.countDistinct("release_row", "chunk_ordinal")
    ).first()
    if tuple(held_counts) != (leakage_stats["cross_split_occurrences"],) * 2:
        raise ValueError("The occurrence hold ledger has missing or duplicate chunk keys.")
    remaining = occurrences.join(
        F.broadcast(hold_keys), ["release_row", "chunk_ordinal"], "left_anti"
    )
    remaining_cross_split = (
        remaining.groupBy("chunk_sha256", "codepoints")
        .agg(F.bit_or("split_bit").alias("split_mask"))
        .where(F.col("split_mask").isin(3, 5, 6, 7))
        .count()
    )
    if remaining_cross_split:
        raise ValueError("An exact cross-split group survived the occurrence holds.")

print("Bound exact chunk leakage audit", leakage_stats)
print(f"Exact cross-split groups after applying the hold ledger: {remaining_cross_split}.")
display(
    exact_groups.where("uncovered_by_parent_overlap")
    .groupBy("long_uncovered")
    .agg(
        F.count("*").alias("groups"),
        F.sum("occurrences").alias("all_occurrences_in_these_groups"),
        F.sum("comparison_occurrences").alias("occurrences_without_known_parent_overlap"),
    )
    .orderBy("long_uncovered")
    .toPandas()
)


## Publish the versioned candidate and audit tables

Keep the risk roster, sample assignment and exact occurrence holds as separate
local Parquet tables. The policy receipt records the reference tokenizer and the
remaining review gates.

In [ ]:
import tempfile

import pyarrow.parquet as pq

policy = {
    **review_policy,
    "status": "scoped_reference_candidate_pending_semantic_gate",
    "nonparagraph_scope": risk_stats,
    "paragraph_scope": structural_stats,
    "samples": sample_stats,
    "exact_leakage": leakage_stats,
    "remaining_exact_cross_split_groups_in_reference_after_holds": remaining_cross_split,
    "leakage_recheck_required_after_any_adopted_boundary_changes": True,
}
tables = {
    "nonparagraph_scope": risk_population.select(
        "release_row",
        "ml_split",
        "format",
        "risk",
        "clean_sha256",
        "clean_chars",
        "nonparagraph_offsets",
        "target_windows",
    ),
    "semantic_sample": sampled,
    "exact_chunk_groups": exact_groups,
    "exact_chunk_holds": holds,
}
if not output.exists():
    with tempfile.TemporaryDirectory(prefix=".chunk-policy-", dir=output.parent) as temp:
        stage = Path(temp) / "snapshot"
        files = {}
        for name, table in tables.items():
            table.write.mode("errorifexists").option("compression", "zstd").parquet(
                str(stage / name)
            )
            files[name] = [
                {
                    "path": path.relative_to(stage).as_posix(),
                    "sha256": sha256_file(path),
                    "rows": pq.ParquetFile(path).metadata.num_rows,
                }
                for path in sorted((stage / name).glob("*.parquet"))
            ]
        policy_path = stage / "CHUNK_POLICY_V1_CANDIDATE.json"
        policy_path.write_text(json.dumps(policy, indent=2) + "\n", encoding="utf-8")
        manifest = {"policy_sha256": sha256_file(policy_path), "tables": files}
        (stage / "MANIFEST.json").write_text(
            json.dumps(manifest, indent=2) + "\n", encoding="utf-8"
        )
        if output.exists():
            raise FileExistsError("A policy snapshot appeared while this one was staged.")
        stage.rename(output)

policy_path = output / "CHUNK_POLICY_V1_CANDIDATE.json"
manifest = json.loads((output / "MANIFEST.json").read_text(encoding="utf-8"))
if (
    sha256_file(policy_path) != manifest["policy_sha256"]
    or json.loads(policy_path.read_text(encoding="utf-8")) != policy
):
    raise ValueError("The saved candidate belongs to another policy, corpus or reference plan.")
for name, entries in manifest["tables"].items():
    upstream["bound_review_table"](output, name, entries)
print(f"Verified local candidate: {output.relative_to(ROOT)}")


## Prepare the review batch

The default `sample-001` batch uses training parents drawn from the frozen samples.
It includes new sentence and whitespace cases, plus paragraph controls with additional
News and Article coverage. Use `FOT08_REVIEW_BATCH=diagnostic` to replay the original
diagnostic tranche. Candidate offsets and parent hashes remain bound to the reference plan.

In [ ]:
import os

old_pilot = ROOT / ".cache/chunk-window-review/0a10356560c83672a868/INDEX.json"
previous_parents = (
    sorted({item["release_row"] for item in json.loads(old_pilot.read_text(encoding="utf-8"))})
    if old_pilot.is_file()
    else []
)
novel = (
    sampled.where(~F.col("release_row").isin(*previous_parents)) if previous_parents else sampled
)
batch_mode = os.environ.get("FOT08_REVIEW_BATCH", "sample-001")
batch_receipt_fields = {}
if batch_mode == "diagnostic":
    trial_order = Window.partitionBy("risk").orderBy("draw_key", "release_row")
    trial = (
        novel.where(F.size("target_windows") <= 4)
        .withColumn("trial_rank", F.row_number().over(trial_order))
        .where(F.col("trial_rank") <= F.when(F.col("sample") == "nonparagraph", 8).otherwise(6))
        .drop("trial_rank")
    )
    batch_rule = "new_parents_up_to_eight_per_risk_and_six_per_control"
elif batch_mode == "sample-001":
    diagnostic_dir = ROOT / ".cache/chunk-policy-review/96e814e5c00fefa5a9ef"
    diagnostic_path = diagnostic_dir / "TRIAL_REPLAY.json"
    diagnostic = json.loads(diagnostic_path.read_text(encoding="utf-8"))
    if (
        diagnostic["policy_manifest_sha256"] != sha256_file(output / "MANIFEST.json")
        or diagnostic["trial_manifest_sha256"] != sha256_file(diagnostic_dir / "MANIFEST.json")
        or diagnostic["units"] != 47
        or len(diagnostic["parents"]) != 33
    ):
        raise ValueError("First sample batch requires the verified diagnostic receipt.")
    diagnosed_parents = [row["release_row"] for row in diagnostic["parents"]]
    quotas = {
        "sentence": 16,
        "whitespace": 12,
        "news_article_paragraph": 8,
        "other_paragraph": 4,
        "terminal_only": 4,
    }
    order = Window.partitionBy("batch_stratum").orderBy("draw_key", "release_row")
    balanced = (
        novel.where(
            (F.col("ml_split") == "train")
            & ~F.col("release_row").isin(*diagnosed_parents)
            & (F.size("target_windows") <= 4)
        )
        .withColumn(
            "batch_stratum",
            F.when(
                (F.col("risk") == "paragraph") & F.col("format").isin("News", "Article"),
                "news_article_paragraph",
            )
            .when(F.col("risk") == "paragraph", "other_paragraph")
            .otherwise(F.col("risk")),
        )
        .withColumn("batch_rank", F.row_number().over(order))
    )
    quota_table = spark.createDataFrame(sorted(quotas.items()), ["batch_stratum", "batch_quota"])
    trial = (
        balanced.join(F.broadcast(quota_table), "batch_stratum")
        .where(F.col("batch_rank") <= F.col("batch_quota"))
        .drop("batch_stratum", "batch_rank", "batch_quota")
    )
    batch_rule = "sample_001_train_sentence_whitespace_and_paragraph_controls"
    batch_receipt_fields = {
        "earlier_diagnostic_replay_sha256": sha256_file(diagnostic_path),
        "batch_quotas": quotas,
    }
else:
    raise ValueError("Select the diagnostic or sample-001 review batch.")

source_rows = (
    trial.join(parents.select("release_row", "clean_text", "paragraphs"), "release_row")
    .join(plans.select("release_row", "plan"), "release_row")
    .orderBy("release_row")
    .collect()
)
if not source_rows or not {row.sample for row in source_rows} == {
    "nonparagraph",
    "paragraph_control",
}:
    raise ValueError("The review batch requires both risk and control parents.")
if batch_mode == "sample-001" and len(source_rows) != sum(quotas.values()):
    raise ValueError("The training sample does not supply the first batch quotas.")

identity = {
    "policy_manifest_sha256": sha256_file(output / "MANIFEST.json"),
    "trial_rule": batch_rule,
    **batch_receipt_fields,
    "trial_parents": [row.release_row for row in source_rows],
    "window_codepoints": CODEPOINT_WINDOW,
    "context_codepoints": CONTEXT_WIDTH,
}
trial_cache = ROOT / ".cache/chunk-policy-review"
trial_cache.mkdir(mode=0o700, parents=True, exist_ok=True)
trial_dir = trial_cache / canonical_sha256(identity)[:20]
by_parent = {}
packets = []
for row in source_rows:
    units = [
        Row(
            unit_ordinal=ordinal,
            start=start,
            end=min(start + CODEPOINT_WINDOW, row.clean_chars),
            context_start=max(0, start - CONTEXT_WIDTH),
            context_end=min(start + CODEPOINT_WINDOW + CONTEXT_WIDTH, row.clean_chars),
        )
        for ordinal, start in enumerate(range(0, row.clean_chars, CODEPOINT_WINDOW))
    ]
    plan_sha = canonical_sha256(
        {
            "reference_manifest_sha256": review_policy["reference_manifest_sha256"],
            "body_sha256": row.clean_sha256,
            "ends": row.plan.ends,
        }
    )
    target = row.nonparagraph_offsets if row.sample == "nonparagraph" else row.plan.ends[:-1]
    target_ordinals = {offset // CODEPOINT_WINDOW for offset in target} if target else {0}
    packet_parent = Row(
        release_row=row.release_row,
        clean_sha256=row.clean_sha256,
        clean_chars=row.clean_chars,
        plan=row.plan,
        plan_sha256=plan_sha,
        review_units=units,
    )
    selected_packets = [
        packet
        for packet in make_review_packets(packet_parent, row.clean_text, row.paragraphs)
        if packet["unit_ordinal"] in target_ordinals
    ]
    if not selected_packets:
        raise ValueError(f"The trial parent has no active review window: {row.release_row}.")
    by_parent[row.release_row] = []
    for packet in selected_packets:
        packets.append(packet)
        targets_in_unit = [x for x in target if packet["start"] <= x < packet["end"]]
        if not set(targets_in_unit) <= set(packet["baseline_boundary_ids"]):
            raise ValueError("A targeted boundary is absent from the reference packet.")
        by_parent[row.release_row].append(
            {
                "line": len(packets),
                "release_row": row.release_row,
                "sample": row.sample,
                "risk": row.risk,
                "format": row.format,
                "packet_sha256": packet["packet_sha256"],
                "target_boundary_ids": targets_in_unit,
            }
        )

lanes = [[], [], []]
for items in sorted(by_parent.values(), key=lambda items: (-len(items), items[0]["line"])):
    slot = min(range(len(lanes)), key=lambda index: (len(lanes[index]), index))
    lanes[slot].extend(items)

if not trial_dir.exists():
    with tempfile.TemporaryDirectory(prefix=".targeted-trial-", dir=trial_cache) as temp:
        stage = Path(temp) / "snapshot"
        stage.mkdir(mode=0o700)
        (stage / "decisions").mkdir(mode=0o700)
        packet_file = stage / "PACKETS.jsonl"
        with packet_file.open("x", encoding="utf-8") as handle:
            for packet in packets:
                handle.write(json.dumps(packet, ensure_ascii=False) + "\n")
        source_sha = sha256_file(packet_file)
        inputs = [packet_file]
        for i, entries in enumerate(lanes, start=1):
            label = f"trial-{i:03d}"
            file = stage / f"BATCH_{i:03d}.json"
            file.write_text(
                json.dumps(
                    {
                        "batch": label,
                        "source_file": packet_file.name,
                        "source_sha256": source_sha,
                        "units": sorted(entries, key=lambda item: item["line"]),
                        "receipt": f"decisions/{label}.jsonl",
                    },
                    indent=2,
                )
                + "\n",
                encoding="utf-8",
            )
            inputs.append(file)
        manifest_path = stage / "MANIFEST.json"
        manifest_path.write_text(
            json.dumps(
                {
                    **identity,
                    "units": len(packets),
                    "files": {file.name: sha256_file(file) for file in inputs},
                },
                indent=2,
            )
            + "\n",
            encoding="utf-8",
        )
        for file in (manifest_path, *inputs):
            file.chmod(0o400)
        if trial_dir.exists():
            raise FileExistsError("A targeted trial appeared during packet preparation.")
        stage.rename(trial_dir)

trial_manifest = json.loads((trial_dir / "MANIFEST.json").read_text(encoding="utf-8"))
if any(trial_manifest.get(key) != value for key, value in identity.items()):
    raise ValueError("The trial directory belongs to another policy or parent roster.")
if any(sha256_file(trial_dir / name) != digest for name, digest in trial_manifest["files"].items()):
    raise ValueError("A trial input changed after publication.")
print(f"Review batch {batch_mode}: {len(packets)} windows for {len(by_parent)} complete parents.")
print(f"Shared trial directory: {trial_dir.relative_to(ROOT)}")


## Build the paragraph-only fallback

Use complete paragraph spans and the reference tokenizer. Preserve indivisible
paragraphs that exceed the budget and keep them out of token-limited model input.
The fallback remains a candidate until adoption and a fresh leakage check.

In [ ]:
import tiktoken

fallback_dir = output.with_name(f"{output.name}_paragraph_fallback")


def paragraph_fallback(text, paragraphs):
    tokenizer = tiktoken.get_encoding(encoding)
    ends = sorted({block.start for block in paragraphs if 0 < block.start < len(text)})
    ends.append(len(text))
    packed_ends, token_counts = [], []
    chunk_start = 0
    previous_end = 0
    previous_count = 0
    for end in ends:
        proposed_count = len(tokenizer.encode_ordinary(text[chunk_start:end]))
        if previous_end > chunk_start and proposed_count > budget:
            packed_ends.append(previous_end)
            token_counts.append(previous_count)
            chunk_start = previous_end
            proposed_count = len(tokenizer.encode_ordinary(text[chunk_start:end]))
        previous_end, previous_count = end, proposed_count
    packed_ends.append(previous_end)
    token_counts.append(previous_count)
    return packed_ends, token_counts


fallback_udf = F.udf(paragraph_fallback, "struct<ends:array<int>,counts:array<int>>")
fallback_contract = {
    "policy_manifest_sha256": sha256_file(output / "MANIFEST.json"),
    "rule": "greedy_complete_paragraph_spans_v1",
    "reference_content_tokens": budget,
}
if not fallback_dir.exists():
    fallback = (
        risk_population.select("release_row", "clean_sha256", "clean_chars", "ml_split")
        .join(
            parents.select("release_row", "clean_sha256", "clean_text", "paragraphs"),
            ["release_row", "clean_sha256"],
        )
        .withColumn("plan", fallback_udf("clean_text", "paragraphs"))
        .withColumn(
            "valid_offsets",
            (F.try_element_at("plan.ends", F.lit(-1)) == F.col("clean_chars"))
            & (F.size("plan.ends") == F.size("plan.counts"))
            & F.forall(
                F.zip_with(
                    "plan.ends",
                    F.slice(F.concat(F.array(F.lit(0)), "plan.ends"), 1, F.size("plan.ends")),
                    lambda end, start: end > start,
                ),
                lambda valid: valid,
            )
            & F.forall("plan.counts", lambda count: count > 0)
            & F.forall(
                F.slice("plan.ends", 1, F.size("plan.ends") - 1),
                lambda end: F.exists("paragraphs", lambda paragraph: paragraph.start == end),
            ),
        )
        .withColumn(
            "over_budget_ordinals",
            F.filter(
                F.sequence(F.lit(0), F.size("plan.ends") - 1),
                lambda i: F.get("plan.counts", i) > budget,
            ),
        )
        .drop("clean_text", "paragraphs")
    )
    with tempfile.TemporaryDirectory(prefix=".paragraph-fallback-", dir=output.parent) as temp:
        stage = Path(temp) / "snapshot"
        fallback.write.mode("errorifexists").option("compression", "zstd").parquet(
            str(stage / "plans")
        )
        result = spark.read.parquet(str(stage / "plans"))
        checks = (
            result.agg(
                F.count("*").alias("parents"),
                F.countDistinct("release_row").alias("unique_parents"),
                F.count_if(~F.coalesce("valid_offsets", F.lit(False))).alias("invalid_parents"),
                F.sum(F.size("over_budget_ordinals")).alias("held_over_budget_chunks"),
                F.count_if(F.size("over_budget_ordinals") > 0).alias(
                    "parents_with_over_budget_chunks"
                ),
                F.sum(F.size("plan.ends")).alias("chunks"),
            )
            .first()
            .asDict()
        )
        if not (
            checks["parents"] == checks["unique_parents"] == risk_stats["parents"]
            and checks["invalid_parents"] == 0
        ):
            raise ValueError(f"Paragraph fallback failed the complete parent checks: {checks}.")
        manifest = {
            **fallback_contract,
            "checks": checks,
            "files": [
                {
                    "path": path.relative_to(stage).as_posix(),
                    "sha256": sha256_file(path),
                    "rows": pq.ParquetFile(path).metadata.num_rows,
                }
                for path in sorted((stage / "plans").glob("*.parquet"))
            ],
        }
        (stage / "MANIFEST.json").write_text(
            json.dumps(manifest, indent=2) + "\n", encoding="utf-8"
        )
        if fallback_dir.exists():
            raise FileExistsError("A fallback snapshot appeared while this one was staged.")
        stage.rename(fallback_dir)
fallback_manifest = json.loads((fallback_dir / "MANIFEST.json").read_text(encoding="utf-8"))
if any(fallback_manifest.get(key) != value for key, value in fallback_contract.items()):
    raise ValueError("The saved fallback belongs to another candidate or policy.")
upstream["bound_review_table"](fallback_dir, "plans", fallback_manifest["files"])
print("Paragraph-only fallback", fallback_manifest["checks"])


## Replay the returned trial decisions

Verify exact packet membership and preserve every undispatched boundary. Retain
material and uncertain evidence findings for the escalation decision.

In [ ]:
from collections import Counter

assignment_roster = {}
for i in range(1, 4):
    assignment = json.loads((trial_dir / f"BATCH_{i:03d}.json").read_text(encoding="utf-8"))
    for item in assignment["units"]:
        digest = item["packet_sha256"]
        if digest in assignment_roster:
            raise ValueError("The trial assigned a packet more than once.")
        assignment_roster[digest] = item

packet_lookup = {packet["packet_sha256"]: packet for packet in packets}
if set(assignment_roster) != set(packet_lookup):
    raise ValueError("The trial assignments do not match the shared packets.")
received = {}
receipt_hashes = {}
for i in range(1, 4):
    label = f"trial-{i:03d}"
    path = trial_dir / "decisions" / f"{label}.jsonl"
    if not path.is_file():
        continue
    assignment = json.loads((trial_dir / f"BATCH_{i:03d}.json").read_text(encoding="utf-8"))
    expected = {unit["packet_sha256"] for unit in assignment["units"]}
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]
    if len(rows) != len(expected) or {row["packet_sha256"] for row in rows} != expected:
        raise ValueError(f"Incorrect final receipt roster for {label}.")
    for decision in rows:
        digest = decision["packet_sha256"]
        packet = packet_lookup[digest]
        item = assignment_roster[digest]
        chosen = decision["selected_boundary_ids"]
        status = decision["status"]
        evidence = decision["evidence_outcome"]
        baseline = packet["baseline_boundary_ids"]
        if (
            not isinstance(chosen, list)
            or any(type(position) is not int for position in chosen)
            or chosen != sorted(set(chosen))
            or not set(chosen) <= set(packet["candidate_boundary_ids"])
            or status not in {"accept", "adjust", "escalate"}
            or evidence not in review_policy["semantic_outcomes"]
            or any(
                not isinstance(decision.get(key), str) or not decision[key].strip()
                for key in ("reason", "reviewer", "reviewer_version")
            )
        ):
            raise ValueError(f"Invalid review decision in {label}.")
        if not (
            (
                status == "accept"
                and chosen == baseline
                and evidence in {"none", "boundary_preference"}
            )
            or (
                status == "adjust"
                and chosen != baseline
                and evidence == "material_evidence_break"
                and set(baseline) - set(chosen) <= set(item["target_boundary_ids"])
            )
            or (
                status == "escalate"
                and not chosen
                and evidence in {"material_evidence_break", "uncertain"}
            )
        ):
            raise ValueError(f"The evidence finding and boundary decision disagree in {label}.")
        received[digest] = decision
    receipt_hashes[label] = sha256_file(path)

encoder = tiktoken.get_encoding(encoding)
outcomes = []
for row in source_rows:
    active = [item["packet_sha256"] for item in by_parent[row.release_row]]
    if any(digest not in received for digest in active):
        continue
    decisions = [received[digest] for digest in active]
    status = "boundary_review_complete"
    ends = set(row.plan.ends)
    counts = []
    if any(decision["status"] == "escalate" for decision in decisions):
        status = "escalated"
    else:
        for digest in active:
            packet = packet_lookup[digest]
            ends.difference_update(packet["baseline_boundary_ids"])
            ends.update(received[digest]["selected_boundary_ids"])
        ends = sorted(ends)
        if ends[-1:] != [row.clean_chars] or ends[0] <= 0:
            raise ValueError("A reviewed plan does not cover its parent.")
        counts = [
            len(encoder.encode_ordinary(row.clean_text[start:end]))
            for start, end in zip([0, *ends[:-1]], ends, strict=True)
        ]
        if any(count < 1 or count > budget for count in counts):
            status = "reference_budget_review_required"
    outcomes.append(
        {
            "release_row": row.release_row,
            "body_sha256": row.clean_sha256,
            "sample": row.sample,
            "ml_split": row.ml_split,
            "risk": row.risk,
            "status": status,
            "internal_end_offsets": ends[:-1] if status == "boundary_review_complete" else None,
            "token_counts": counts if status == "boundary_review_complete" else None,
            "evidence_outcomes": sorted({decision["evidence_outcome"] for decision in decisions}),
        }
    )
print(f"Received {len(received)} of {len(packets)} targeted trial decisions.")
if outcomes:
    counts_by_sample = Counter(
        (assignment_roster[digest]["sample"], decision["status"], decision["evidence_outcome"])
        for digest, decision in received.items()
    )
    display(
        pd.DataFrame(
            [(*key, value) for key, value in sorted(counts_by_sample.items())],
            columns=["Sample", "Decision", "Evidence finding", "Windows"],
        )
    )
    display(
        pd.DataFrame(outcomes).groupby(["sample", "status"]).size().rename("Parents").to_frame()
    )

if len(received) == len(packets):
    result = {
        "policy_manifest_sha256": sha256_file(output / "MANIFEST.json"),
        "trial_manifest_sha256": sha256_file(trial_dir / "MANIFEST.json"),
        "receipts": receipt_hashes,
        "units": len(received),
        "parents": outcomes,
        "status": (
            "targeted_trial_complete_full_samples_pending"
            if batch_mode == "diagnostic"
            else "sample_batch_001_complete_full_samples_pending"
        ),
    }
    report_name = "TRIAL_REPLAY.json" if batch_mode == "diagnostic" else "BATCH_REPLAY.json"
    trial_report = trial_dir / report_name
    if trial_report.exists():
        if json.loads(trial_report.read_text(encoding="utf-8")) != result:
            raise ValueError("The targeted trial replay changed.")
    else:
        with tempfile.NamedTemporaryFile(
            dir=trial_dir, mode="w", encoding="utf-8", delete=False
        ) as handle:
            handle.write(json.dumps(result, indent=2) + "\n")
            staged = Path(handle.name)
        staged.rename(trial_report)
    print(f"Targeted trial replay: {trial_report.relative_to(ROOT)}")


## Check exact reuse of newly formed slices

Compare the first sample batch's changed slices against the unchanged validation
and test reference plans. This check covers the new training slices from verified
parent plans. The full leakage gate still applies before adopting any revised plan.

In [ ]:
if batch_mode == "sample-001" and len(received) == len(packets):
    reference_rows = {row.release_row: row for row in source_rows}
    introduced_slices = []
    for outcome in outcomes:
        if outcome["status"] != "boundary_review_complete":
            continue
        row = reference_rows[outcome["release_row"]]
        baseline_ends = row.plan.ends
        candidate_ends = [*outcome["internal_end_offsets"], row.clean_chars]
        baseline_spans = set(zip([0, *baseline_ends[:-1]], baseline_ends, strict=True))
        candidate_spans = zip([0, *candidate_ends[:-1]], candidate_ends, strict=True)
        for ordinal, (start, end) in enumerate(candidate_spans):
            if (start, end) not in baseline_spans:
                introduced_slices.append((row.release_row, ordinal, start, end))

    matched_pairs = []
    if introduced_slices:
        source_text = parents.select("release_row", "clean_text")
        introduced = (
            spark.createDataFrame(
                introduced_slices, "release_row long, ordinal int, start int, end int"
            )
            .join(source_text, "release_row")
            .select(
                "release_row",
                "ordinal",
                "start",
                "end",
                (F.col("end") - F.col("start")).alias("codepoints"),
                F.sha2(
                    F.substring("clean_text", F.col("start") + 1, F.col("end") - F.col("start")),
                    256,
                ).alias("chunk_sha256"),
            )
        )
        if introduced.count() != len(introduced_slices):
            raise ValueError("A revised slice could not be bound to its original parent.")
        heldout_positions = (
            plans.where(F.col("ml_split").isin("validation", "test"))
            .select(
                "release_row",
                "ml_split",
                "plan.ends",
                F.posexplode("plan.ends").alias("ordinal", "end"),
            )
            .withColumn("start", F.coalesce(F.get("ends", F.col("ordinal") - 1), F.lit(0)))
            .withColumn("codepoints", F.col("end") - F.col("start"))
            .drop("ends")
        )
        changed_lengths = introduced.select("codepoints").distinct()
        heldout_hashes = (
            heldout_positions.join(F.broadcast(changed_lengths), "codepoints")
            .join(source_text, "release_row")
            .select(
                F.col("release_row").alias("heldout_release_row"),
                F.col("ordinal").alias("heldout_ordinal"),
                F.col("ml_split").alias("heldout_split"),
                "codepoints",
                F.sha2(F.substring("clean_text", F.col("start") + 1, "codepoints"), 256).alias(
                    "chunk_sha256"
                ),
            )
        )
        matched_pairs = [
            row.asDict()
            for row in heldout_hashes.join(F.broadcast(introduced), ["chunk_sha256", "codepoints"])
            .orderBy("release_row", "ordinal", "heldout_release_row", "heldout_ordinal")
            .collect()
        ]
    delta = {
        "reference_plan_manifest_sha256": review_policy["reference_manifest_sha256"],
        "batch_replay_sha256": sha256_file(trial_report),
        "scope": "new_batch_slices_against_unmodified_validation_and_test_reference",
        "new_slices": len(introduced_slices),
        "matched_pairs": matched_pairs,
    }
    audit_path = trial_dir / "EXACT_DELTA.json"
    if audit_path.exists():
        if json.loads(audit_path.read_text(encoding="utf-8")) != delta:
            raise ValueError("The revised-slice audit changed.")
    else:
        with tempfile.NamedTemporaryFile(
            dir=trial_dir, mode="w", encoding="utf-8", delete=False
        ) as handle:
            handle.write(json.dumps(delta, indent=2) + "\n")
            staged = Path(handle.name)
        staged.rename(audit_path)
    print(
        f"Compared {len(introduced_slices)} newly formed slices against the held-out "
        f"reference. Exact matching pairs: {len(matched_pairs)}."
    )
